# HS071 Optimization Problem

For a solution to the HS071 constrained function minimization problem using a Python script instead of a notebook, see the [HS071 Python script documentation](../scripts/hs071.rst).

## Description

HS071 is Problem 71 from the collection of nonlinear programming test problems by Hock and Schittkowski <cite data-footcite="Hoch:1981">(1981)</cite>, who cite Bartholomew-Biggs <cite data-footcite="Bartholomew-Biggs:1976">(1976)</cite> as the original source. The problem is to minimize the function
$$
   f(x)=x_{0} x_{3}\left(x_{0}+x_{1}+x_{2}\right)+x_{2}
$$
subject to the nonlinear constraints
$$
\begin{aligned}
   x_{0} x_{1} x_{2} x_{3} &\geq 25 \\
   x_{0}^{2}+x_{1}^{2}+x_{2}^{2}+x_{3}^{2} & = 40
\end{aligned}
$$
and the variable bounds
$$
1 \leq x_{i} \leq 5, \quad i=0, 1, 2, 3
$$
The initial guess is given by
$$
\boldsymbol{x} = [1,5,5,1]^T
$$

(Note that we have used 0-based indexing in the problem statement, whereas Hock and Schittkowski used 1-based indexing.)

## YAPSS Solution

First, we instantiate the problem. It has no phases, 4 parameters, and 2 discrete constraints:

Name the four variables and the two constraints, and declare that the problem has no phases:

In [ ]:
import yapss


class Parameter(yapss.Vector):
    x = yapss.field(size=4)  # design variables


class Discrete(yapss.Vector):
    product = yapss.field()  # the product of all four, at least 25
    sum_of_squares = yapss.field()  # the sum of their squares, exactly 40


class Phases(yapss.Phases):
    """None. The problem has no trajectory, so it has no phases."""

In [ ]:
problem = yapss.Problem("HS071", phases=Phases, parameter=Parameter, discrete=Discrete)

We then define the objective and discrete constraint callback functions:

In [ ]:
@problem.register.objective
def objective(arg):
    x = arg.parameter.x
    return x[0] * x[3] * (x[0] + x[1] + x[2]) + x[2]


@problem.register.discrete
def discrete(arg, out):
    x = arg.parameter.x
    out.discrete.product = x[0] * x[1] * x[2] * x[3]
    out.discrete.sum_of_squares = x[0] ** 2 + x[1] ** 2 + x[2] ** 2 + x[3] ** 2
    return out

Set the bounds on the parameters and the constraint functions, per the problem statement:

In [ ]:
problem.parameter.bounds.x[:] = (1.0, 5.0)
problem.discrete.bounds.product = (25.0, None)
problem.discrete.bounds.sum_of_squares = (40.0, 40.0)

We also provide an initial guess for the parameter values:

In [ ]:
problem.parameter.guess.x[:] = [1.0, 5.0, 5.0, 1.0]

Specify the YAPSS and Ipopt options:

In [ ]:
problem.derivatives.order = "second"
problem.derivatives.method = "auto"
problem.ipopt_options.print_user_options = "no"
problem.ipopt_options.print_level = 3

Solve the problem:

In [ ]:
solution = problem.solve()

Finally, we format and print the solution:

In [ ]:
print("Solution of the primal variables, x")
for i, value in enumerate(solution.parameter.x):
    print(f"x[{i}] = {value:1.6e}")

print("\nSolution of the constraint multipliers, lambda")
print(f"lambda[product]        = {solution.discrete_multiplier.product:1.6e}")
print(f"lambda[sum_of_squares] = {solution.discrete_multiplier.sum_of_squares:1.6e}")

print("\nConstraint values")
print(f"product        = {solution.discrete.product:1.6e}")
print(f"sum of squares = {solution.discrete.sum_of_squares:1.6e}")

print("\nObjective value")
print(f"f(x*) = {solution.objective:1.6e}")

The solution above replicates the example given (in C++) in the [Ipopt interface documentation](https://coin-or.github.io/Ipopt/INTERFACES.html#INTERFACE_CPP), in much less code.

## References